<a href="https://colab.research.google.com/github/mtarazona23/PROGCOM-B/blob/main/juego_huecos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import random
import ipywidgets as widgets
from IPython.display import display

HUECOS = 24
nombres = ["", ""]
rellenos = [0, 0]
turno = 0
eleccion1 = ""
fase = "inicio"

nombre1 = widgets.Text(value="Jugador 1", description="Jugador 1:")
nombre2 = widgets.Text(value="Jugador 2", description="Jugador 2:")
pc = widgets.Checkbox(description="Jugar contra la computadora")
btn_empezar = widgets.Button(description="Empezar")
btn_piedra = widgets.Button(description="✊ Piedra")
btn_papel = widgets.Button(description="✋ Papel")
btn_tijera = widgets.Button(description="✌️ Tijera")
btn_dado = widgets.Button(description=" Lanzar dado", disabled=True)
mensaje = widgets.HTML()
tablero = widgets.HTML()

def dibujar():
    texto = ""
    for i in range(2):
        t = "●" * rellenos[i] + "○" * (HUECOS - rellenos[i])
        secciones = []
        for s in range(0, HUECOS, 4):
            secciones.append(t[s:s + 4])
        texto += f"<b>{nombres[i]}</b>: <span style='font-size:22px'>{' | '.join(secciones)}</span> ({rellenos[i]}/{HUECOS})<br>"
    tablero.value = texto

def empezar(b):
    global turno, eleccion1, fase
    nombres[0] = nombre1.value
    nombres[1] = "Computadora" if pc.value else nombre2.value
    rellenos[0] = 0
    rellenos[1] = 0
    eleccion1 = ""
    fase = "ppt"
    btn_dado.disabled = True
    mensaje.value = f"Piedra, papel o tijera: <b>{nombres[0]}</b>, elige"
    dibujar()

def elegir(opcion):
    global eleccion1
    if fase != "ppt":
        return
    if pc.value:
        resolver(opcion, random.choice(["piedra", "papel", "tijera"]))
    elif eleccion1 == "":
        eleccion1 = opcion
        mensaje.value = f"Ahora elige <b>{nombres[1]}</b> (sin que {nombres[0]} mire)"
    else:
        resolver(eleccion1, opcion)

def resolver(e1, e2):
    global turno, eleccion1, fase
    eleccion1 = ""
    texto = f"{nombres[0]} eligió {e1} y {nombres[1]} eligió {e2}. "
    if e1 == e2:
        mensaje.value = texto + f"¡Empate! Otra vez, <b>{nombres[0]}</b> elige"
        return
    if (e1 == "piedra" and e2 == "tijera") or (e1 == "papel" and e2 == "piedra") or (e1 == "tijera" and e2 == "papel"):
        turno = 0
    else:
        turno = 1
    fase = "juego"
    btn_dado.disabled = False
    texto += f"<b>{nombres[turno]}</b> empieza!"
    if pc.value and turno == 1:
        texto += "<br>" + jugar_turno()
    mensaje.value = texto + siguiente()


def jugar_turno():
    global turno, fase
    dado = random.randint(1, 6)
    texto = f" {nombres[turno]} sacó <b>{dado}</b> y rellena {dado} huecos."
    rellenos[turno] = min(HUECOS, rellenos[turno] + dado)
    dibujar()
    if rellenos[turno] == HUECOS:
        fase = "fin"
        btn_dado.disabled = True
        return texto + f"<br><b style='font-size:20px'> ¡{nombres[turno]} ganó!</b>"
    turno = 1 - turno
    return texto

def siguiente():
    if fase == "juego":
        return f"<br>Turno de <b>{nombres[turno]}</b>"
    return ""

def lanzar(b):
    if fase != "juego":
        return
    texto = jugar_turno()
    if fase == "juego" and pc.value and turno == 1:
        texto += "<br>" + jugar_turno()
    mensaje.value = texto + siguiente()

btn_empezar.on_click(empezar)
btn_piedra.on_click(lambda b: elegir("piedra"))
btn_papel.on_click(lambda b: elegir("papel"))
btn_tijera.on_click(lambda b: elegir("tijera"))
btn_dado.on_click(lanzar)

display(widgets.VBox([
    widgets.HTML("<h2> Juego de los Huecos</h2>"),
    nombre1, nombre2, pc, btn_empezar,
    widgets.HBox([btn_piedra, btn_papel, btn_tijera]),
    btn_dado, mensaje, tablero
]))